In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))

In [ ]:
# 安装 diffusers + transformers + huggingface_hub 等（可能需几分钟）
!pip install -q diffusers==0.20.0 transformers accelerate safetensors huggingface_hub

In [ ]:
# 升级 diffusers/transformers/huggingface_hub/accelerate/safetensors
!pip install -q --upgrade diffusers transformers huggingface_hub accelerate safetensors
# 然后检查版本
import huggingface_hub, diffusers, transformers
print("huggingface_hub", huggingface_hub.__version__)
print("diffusers", diffusers.__version__)
print("transformers", transformers.__version__)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# 下面的 model_id 我选了 runwayml 的 v1-5（稳定且常用）
model_id = "runwayml/stable-diffusion-v1-5"

# 从 hub 加载模型（如果报错提示权限，请去模型页面同意条款）
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")  # 把模型移动到 GPU
pipe.enable_attention_slicing()  # 降低显存峰值，避免 OOM


In [ ]:
prompt = "a cyberpunk city with neon lights, ultra detailed, cinematic, 4k"
generator = torch.Generator(device="cuda").manual_seed(42)  # 固定随机种子，方便复现
image = pipe(prompt, guidance_scale=7.5, num_inference_steps=30, generator=generator).images[0]

# 显示并保存
display(image)
image.save("/content/result.png")
print("Saved /content/result.png")


In [ ]:
prompts = [
    "a kawaii anime girl listening to music, colorful, high detail",
    "a wooden cabin in snowy mountains during sunrise, photo realistic",
    "a futuristic robot playing guitar in a neon-lit bar, cinematic"
]

images = pipe(prompts, guidance_scale=7.5, num_inference_steps=25).images
for i, img in enumerate(images):
    path = f"/content/result_{i}.png"
    img.save(path)
    print("Saved", path)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ai-text2img/results
# 移动刚生成的文件到 Drive
!cp /content/result*.png /content/drive/MyDrive/ai-text2img/results/
print("Copied generated images to your Google Drive at MyDrive/ai-text2img/results")


In [ ]:
!zip -r results.zip /content/result*.png
from google.colab import files
files.download("results.zip")

In [ ]:
prompt = "a majestic lion sitting on a throne, detailed portrait, cinematic lighting"
negative_prompt = "blurry, lowres, text, watermark, deformed"
image = pipe(prompt, negative_prompt=negative_prompt, guidance_scale=8.0, num_inference_steps=30, height=512, width=512).images[0]
display(image)
image.save("/content/result_negative.png")